<a href="https://colab.research.google.com/github/Viswas-Vinayakumar/Cloudnotes/blob/main/LCP3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Food parsing: error analysis

A structured extraction task. Input is free-text meal logs.
Output is JSON with item, quantity, unit.

This notebook measures how well the parser works and why it fails.
The parser itself is five lines. The measurement is the point.

Model: openai/gpt-oss-120b via Groq. temperature=0 for repeatable runs.

Three steps: prompt template -> model -> JSON parser.
Kept deliberately simple so failures come from the prompt

In [1]:
!pip install -qU langchain-groq

from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

llm = ChatGroq(
    api_key=userdata.get('langchaingroqapi'),
    model="openai/gpt-oss-120b",
    temperature=0,
)

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Extract foods from the text. Return JSON with a list called items. "
     "Each item has: item, quantity, unit. Use null if not stated. "
     "Return only JSON, no other text."),
    ("human", "{text}"),
])

chain = prompt | llm | JsonOutputParser()

tests = [
    "200g magerquark",
    "2 boiled eggs",
    "a handful of almonds",
    "2 Eier und 200g Magerquark",
    "skipped breakfast today",
]

for t in tests:
    print(t)
    print("   ->", chain.invoke({"text": t}))
    print()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.9 MB/s eta 0:00:00
200g magerquark
   -> {'items': [{'item': 'magerquark', 'quantity': 200, 'unit': 'g'}]}

2 boiled eggs
   -> {'items': [{'item': 'boiled eggs', 'quantity': 2, 'unit': None}]}

a handful of almonds
   -> {'items': [{'item': 'almonds', 'quantity': 'handful', 'unit': None}]}

2 Eier und 200g Magerquark
   -> {'items': [{'item': 'Eier', 'quantity': 2, 'unit': None}, {'item': 'Magerquark', 'quantity': 200, 'unit': 'g'}]}

skipped breakfast today
   -> {'items': []}



## Test cases

Five inputs, chosen to cover different input shapes:
clean, countable, vague quantity, German, no food.
Four of the five are expected to expose a problem.

## Findings

| # | Input | Result | What went wrong |
|---|-------|--------|-----------------|
| 1 | 200g magerquark | pass | - |
| 2 | 2 boiled eggs | fail | plural name, includes cooking method |
| 3 | a handful of almonds | fail | quantity is text, not a number |
| 4 | 2 Eier und 200g Magerquark | fail | capitalised, German word kept |
| 5 | skipped breakfast today | pass | - |

All four failures trace back to rules missing from the prompt.
The prompt never asked for lowercase, singular names, English
translation, or numeric quantities.

Failure 3 is the most serious. A text quantity will crash any
code that does arithmetic on it. The others are cosmetic.

In [2]:
from typing import List, Optional
from pydantic import BaseModel, Field

class FoodItem(BaseModel):
    item: str = Field(
        description="food name, lowercase, singular, English"
    )
    quantity: Optional[float] = Field(
        default=None,
        description="a number only. null if no amount was stated"
    )
    unit: Optional[str] = Field(
        default=None,
        description="one of: g, ml, piece, scoop, tbsp. null if not stated"
    )

class ParsedEntry(BaseModel):
    items: List[FoodItem]


structured_llm = llm.with_structured_output(ParsedEntry)
chain2 = prompt | structured_llm

for t in tests:
    print(t)
    print("   ->", chain2.invoke({"text": t}))
    print()

200g magerquark
   -> items=[FoodItem(item='quark', quantity=200.0, unit='g')]

2 boiled eggs
   -> items=[FoodItem(item='egg', quantity=2.0, unit=None)]

a handful of almonds
   -> items=[FoodItem(item='almond', quantity=None, unit='handful')]

2 Eier und 200g Magerquark
   -> items=[FoodItem(item='egg', quantity=2.0, unit=None), FoodItem(item='quark', quantity=200.0, unit='g')]

skipped breakfast today
   -> items=[]



## Scoreboard: prompt-only vs Pydantic schema

| # | Input | Before | After |
|---|-------|--------|-------|
| 1 | 200g magerquark | pass | **fail** — became `quark`, fat level lost |
| 2 | 2 boiled eggs | fail — plural name, cooking method kept | fail — unit missing, should be `piece` |
| 3 | a handful of almonds | fail — `quantity='handful'` (text) | fail — text moved to `unit='handful'` |
| 4 | 2 Eier und 200g Magerquark | fail — German kept, capitalised | **pass** |
| 5 | skipped breakfast today | pass | pass |

**2/5 before, 2/5 after.** The total did not move, but three
different things happened underneath it: two failures fixed,
one new failure created, one failure relocated to another field.

Constraining `quantity` to a number did not delete the bad value.
It pushed it into `unit`, the neighbouring field that still
accepted text. Narrowing one field moves bad data to whichever
field is still loose.

This is why a single pass rate is not enough. A flat score hid
three separate changes.